# Paper 001: minimal AAD experiment workflow

Run one stage at a time: setup, dataset check, model check, then experiments.
Failures stream their complete output in the cell. Test evaluation is not part
of this notebook.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/sanelehlabisa/phd_research.git"
REPOSITORY_ROOT = "/content/phd_research"
IMPLEMENTATION_DIR = (
    f"{REPOSITORY_ROOT}/papers/"
    "001-journal-abnormal-activity-recognition/implementation"
)
KAGGLE_DATASET_HANDLE = "sanelehlabisa/abnormal-activities-dataset"
KAGGLE_DATASET_FOLDER = "abnormal-activities-dataset"
DATASET_DIR_OVERRIDE = ""
RUNS_DIR = f"{IMPLEMENTATION_DIR}/runs"

SEQUENCE_LENGTH = 16
HEIGHT = 32
WIDTH = 32
SEED = 42
MODEL_LAYERS = [(8, (3, 3)), (16, (3, 3))]
EXPERIMENT_STAGE = "architecture-screen"

# Enable only the stage being run.
RUN_COLAB_SETUP = False
RUN_DATASET_CHECK = False
RUN_MODEL_CHECK = False
RUN_EXPERIMENTS = False

In [ ]:
import importlib.util
import os
import shlex
import subprocess
import sys


def run_command(command: list[str], cwd: Path | None = None) -> None:
    """Run a command and stream its complete output into this cell."""
    command = [str(part) for part in command]
    print("$", shlex.join(command), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Command output stream is unavailable")
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")


def implementation_path() -> Path:
    """Return the cloned implementation directory."""
    path = Path(IMPLEMENTATION_DIR).resolve()
    if not (path / "src").is_dir():
        raise FileNotFoundError(f"Implementation not found: {path}; run setup first")
    return path


def dataset_path() -> Path:
    """Return an override or the newest KaggleHub-cached AAD folder."""
    if DATASET_DIR_OVERRIDE.strip():
        path = Path(DATASET_DIR_OVERRIDE).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(f"Dataset override not found: {path}")
        return path
    owner, dataset_name = KAGGLE_DATASET_HANDLE.split("/", maxsplit=1)
    versions = (
        Path("/root/.cache/kagglehub/datasets")
        / owner
        / dataset_name
        / "versions"
    )
    candidates = list(versions.glob(f"*/{KAGGLE_DATASET_FOLDER}"))
    candidates = [path for path in candidates if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("Cached AAD folder not found; run setup first")
    return max(
        candidates,
        key=lambda path: int(path.parent.name) if path.parent.name.isdigit() else -1,
    ).resolve()

## 1. One-time setup

Set RUN_COLAB_SETUP=True, run this cell, then set it back to False. The
cell is safe to rerun: it updates an existing clone and reuses KaggleHub's cache.

In [ ]:
if RUN_COLAB_SETUP:
    repository = Path(REPOSITORY_ROOT).resolve()
    if (repository / ".git").is_dir():
        run_command(["git", "pull", "--ff-only"], cwd=repository)
    elif repository.exists():
        raise FileExistsError(f"Repository path exists but is not a clone: {repository}")
    else:
        repository.parent.mkdir(parents=True, exist_ok=True)
        run_command(["git", "clone", REPOSITORY_URL, str(repository)])

    packages = {
        "av": "av==16.1.0",
        "cv2": "opencv-python",
        "kagglehub": "kagglehub",
        "matplotlib": "matplotlib",
        "numpy": "numpy",
        "pandas": "pandas",
        "PIL": "pillow",
        "sklearn": "scikit-learn",
        "torchmetrics": "torchmetrics==1.8.2",
        "tqdm": "tqdm",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        run_command([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("All required packages are available.")

    import kagglehub

    path = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
    print("Path to dataset files:", path)
    downloaded_dataset = (Path(path) / KAGGLE_DATASET_FOLDER).resolve()
    if not downloaded_dataset.is_dir():
        raise FileNotFoundError(f"AAD folder not found after download: {downloaded_dataset}")
    print("AAD dataset:", downloaded_dataset)
    print("CUDA check will run with the model stage.")
else:
    print("Setup disabled.")

## 2. Check one real dataset sample

Set RUN_DATASET_CHECK=True. This loads and augments one clip directly; it
does not encode preview videos or hide errors inside a child process.

In [ ]:
if RUN_DATASET_CHECK:
    implementation = implementation_path()
    os.chdir(implementation)
    if str(implementation) not in sys.path:
        sys.path.insert(0, str(implementation))

    from src.dataset import AHARDataset, VideoAugmentation

    dataset = AHARDataset(
        dataset_path(),
        sequence_length=SEQUENCE_LENGTH,
        frame_size=(HEIGHT, WIDTH),
    )
    if not dataset.samples:
        raise ValueError("AAD contains no supported video samples")
    sample_video, sample_label = dataset[0]
    augmented_video = VideoAugmentation()(sample_video.clone())
    print("Dataset:", dataset.dataset_dir)
    print("Samples:", len(dataset), "Classes:", dataset.class_names)
    print("Sample class:", dataset.class_names[sample_label])
    print("Clean shape/range:", tuple(sample_video.shape), float(sample_video.min()), float(sample_video.max()))
    print("Augmented shape/range:", tuple(augmented_video.shape), float(augmented_video.min()), float(augmented_video.max()))
else:
    print("Dataset check disabled.")

## 3. Check the model and approved architecture list

After the dataset check succeeds, set RUN_MODEL_CHECK=True. This runs one
real clip through a small custom ConvLSTM and lists the frozen candidates.

In [ ]:
if RUN_MODEL_CHECK:
    if "sample_video" not in globals() or "dataset" not in globals():
        raise RuntimeError("Run the dataset check cell first")
    import torch
    from src.model import CustomConvLSTM, count_trainable_parameters

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CustomConvLSTM(dataset.num_classes, layers=MODEL_LAYERS).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(sample_video.unsqueeze(0).to(device))
    print("Device:", device)
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    print(model)
    print("Trainable parameters:", count_trainable_parameters(model))
    print("Output shape:", tuple(logits.shape))

    implementation = implementation_path()
    run_command(
        [
            sys.executable,
            "-u",
            "-m",
            "src.experiments",
            "--config",
            str(implementation / "configs/aad_screening_reference.json"),
            "--candidates-config",
            str(implementation / "configs/aad_architecture_candidates.json"),
            "--list-models",
        ],
        cwd=implementation,
    )
    del model, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Model check disabled.")

## 4. Run one controlled experiment stage

Start with EXPERIMENT_STAGE=architecture-screen. Set RUN_EXPERIMENTS=True
only after the dataset and model checks pass. Output is streamed live and
selection remains validation-only.

In [ ]:
if RUN_EXPERIMENTS:
    allowed_stages = {
        "architecture-screen",
        "baseline-confirmation",
        "published-topology",
        "focused-ablations",
    }
    if EXPERIMENT_STAGE not in allowed_stages:
        raise ValueError(f"Unknown experiment stage: {EXPERIMENT_STAGE}")
    implementation = implementation_path()
    runs = Path(RUNS_DIR).resolve()
    runs.mkdir(parents=True, exist_ok=True)
    run_command(
        [
            sys.executable,
            "-u",
            "-m",
            "src.experiments",
            "--plan-config",
            str(implementation / "configs/aad_controlled_experiment_plan.json"),
            "--run-plan-stage",
            EXPERIMENT_STAGE,
            "--plan-dataset-dir",
            str(dataset_path()),
            "--plan-runs-dir",
            str(runs),
        ],
        cwd=implementation,
    )
else:
    print("Experiments disabled.")

Experiment artifacts are written under implementation/runs/. Download the
completed run if you want to retain it after the Colab runtime ends. Do not use
test results to choose an architecture.